In [ ]:
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


class Config:
    def __init__(self):
        self.root_dir = "/content/unprocessed_dataset/"
        self.model_path = "/content/face_landmarker.task"
        self.detection_threshold = 0.2

        self.mouth_width_scale = 1.20
        self.mouth_height_scale = 1.10
        self.mouth_vertical_shift = 0.08

        self.num_random_frames_per_video = 3
        self.random_seed = 42


class FaceMouthLandmarkDetector:
    def __init__(self, model_path, threshold=0.2):
        self.threshold = threshold

        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_faces=1,
            min_face_detection_confidence=threshold,
            min_face_presence_confidence=threshold,
            min_tracking_confidence=threshold,
            output_face_blendshapes=False,
            output_facial_transformation_matrixes=False
        )
        self.landmarker = vision.FaceLandmarker.create_from_options(options)

        self.mouth_indices = [
            61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
            78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
            191, 80, 81, 82, 13, 312, 311, 310, 415
        ]

    def detect(self, frame_bgr):
        try:
            h, w = frame_bgr.shape[:2]

            if len(frame_bgr.shape) == 2:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
            else:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result = self.landmarker.detect(mp_image)
        except Exception:
            return None

        if not result.face_landmarks:
            return None

        face_landmarks = result.face_landmarks[0]

        def pt(idx):
            return (
                float(face_landmarks[idx].x * w),
                float(face_landmarks[idx].y * h)
            )

        xs = [lm.x * w for lm in face_landmarks]
        ys = [lm.y * h for lm in face_landmarks]

        x1 = float(min(xs))
        y1 = float(min(ys))
        x2 = float(max(xs))
        y2 = float(max(ys))

        mouth_points = [pt(idx) for idx in self.mouth_indices]

        return {
            "score": 1.0,
            "bbox": [x1, y1, x2, y2],
            "landmarks": {
                "left_eye": pt(33),
                "right_eye": pt(263),
                "nose": pt(1),
                "mouth_left": pt(61),
                "mouth_right": pt(291),
                "mouth_points": mouth_points,
            }
        }


def crop_with_reflect_padding(img, x1, y1, x2, y2):
    h, w = img.shape[:2]

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)

    if pad_left or pad_top or pad_right or pad_bottom:
        img = cv2.copyMakeBorder(
            img,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            borderType=cv2.BORDER_REFLECT
        )

    x1 += pad_left
    x2 += pad_left
    y1 += pad_top
    y2 += pad_top

    return img[y1:y2, x1:x2]


def is_valid_mouth(landmarks):
    points = landmarks.get("mouth_points")
    nose = np.array(landmarks["nose"], dtype=np.float32)

    if not points or len(points) < 4:
        return False

    pts = np.array(points, dtype=np.float32)

    min_xy = pts.min(axis=0)
    max_xy = pts.max(axis=0)

    width = float(max_xy[0] - min_xy[0])
    height = float(max_xy[1] - min_xy[1])
    center = pts.mean(axis=0)

    mouth_to_nose = np.linalg.norm(center - nose)

    if width < 10:
        return False
    if height < 4:
        return False
    if mouth_to_nose < 5:
        return False
    if height > width * 1.2:
        return False

    return True


def compute_mouth_box_from_landmarks(landmarks, cfg):
    pts = np.array(landmarks["mouth_points"], dtype=np.float32)

    min_xy = pts.min(axis=0)
    max_xy = pts.max(axis=0)

    x1_raw, y1_raw = min_xy
    x2_raw, y2_raw = max_xy

    width = x2_raw - x1_raw
    height = y2_raw - y1_raw

    cx = (x1_raw + x2_raw) / 2.0
    cy = (y1_raw + y2_raw) / 2.0 - cfg.mouth_vertical_shift * width

    crop_w = max(int(width * cfg.mouth_width_scale), 12)
    crop_h = max(int(height * cfg.mouth_height_scale), 8)

    x1 = int(cx - crop_w / 2)
    y1 = int(cy - crop_h / 2)
    x2 = x1 + crop_w
    y2 = y1 + crop_h

    return x1, y1, x2, y2


def extract_crop(frame, box):
    if box is None:
        return None

    x1, y1, x2, y2 = box

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    crop = crop_with_reflect_padding(gray, x1, y1, x2, y2)

    if crop.size == 0:
        return None

    return crop


def draw_box(frame, box, mouth_points):
    if len(frame.shape) == 2:
        vis = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    else:
        vis = frame.copy()

    for x, y in mouth_points:
        cv2.circle(vis, (int(x), int(y)), 1, (0, 255, 0), -1)

    x1, y1, x2, y2 = box
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)

    return vis


def collect_train_videos(root_dir):
    items = []

    for person in sorted(os.listdir(root_dir)):
        person_path = os.path.join(root_dir, person)
        if not os.path.isdir(person_path):
            continue

        for condition in sorted(os.listdir(person_path)):
            condition_path = os.path.join(person_path, condition)
            if not os.path.isdir(condition_path):
                continue

            for file_name in sorted(os.listdir(condition_path)):
                if not file_name.lower().endswith(".avi"):
                    continue

                video_path = os.path.join(condition_path, file_name)
                video_name = os.path.splitext(file_name)[0]

                items.append({
                    "person": person,
                    "condition": condition,
                    "video_name": video_name,
                    "video_path": video_path
                })

    return items


def sample_random_frames_from_video(video_path, num_frames, rng):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    sample_count = min(num_frames, total_frames)
    frame_indices = sorted(rng.sample(range(total_frames), sample_count))

    frames = []
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            frames.append((frame_idx, frame))

    cap.release()
    return frames


def preview_all_train_videos(cfg):
    rng = random.Random(cfg.random_seed)
    detector = FaceMouthLandmarkDetector(
        model_path=cfg.model_path,
        threshold=cfg.detection_threshold
    )

    videos = collect_train_videos(cfg.root_dir)
    if not videos:
        raise RuntimeError("No .avi videos found in train folder structure")

    results = []

    for item in videos:
        sampled_frames = sample_random_frames_from_video(
            item["video_path"],
            cfg.num_random_frames_per_video,
            rng
        )

        for frame_idx, frame in sampled_frames:
            face = detector.detect(frame)

            if face is None or not is_valid_mouth(face["landmarks"]):
                results.append({
                    "title": f"{item['person']} | {item['condition']} | {item['video_name']} | frame {frame_idx}",
                    "frame_vis": frame if len(frame.shape) == 3 else cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR),
                    "crop": None,
                    "status": "invalid_or_not_detected"
                })
                continue

            box = compute_mouth_box_from_landmarks(face["landmarks"], cfg)
            crop = extract_crop(frame, box)
            frame_vis = draw_box(frame, box, face["landmarks"]["mouth_points"])

            results.append({
                "title": f"{item['person']} | {item['condition']} | {item['video_name']} | frame {frame_idx}",
                "frame_vis": frame_vis,
                "crop": crop,
                "status": "ok"
            })

    if not results:
        print("No preview results")
        return

    rows = len(results)
    plt.figure(figsize=(10, 4 * rows))

    plot_idx = 1
    for item in results:
        plt.subplot(rows, 2, plot_idx)
        plt.imshow(cv2.cvtColor(item["frame_vis"], cv2.COLOR_BGR2RGB))
        plt.title(item["title"] + "\n" + item["status"])
        plt.axis("off")
        plot_idx += 1

        plt.subplot(rows, 2, plot_idx)
        if item["crop"] is not None:
            plt.imshow(item["crop"], cmap="gray")
            plt.title("mouth crop")
        else:
            plt.imshow(np.full((40, 80), 128, dtype=np.uint8), cmap="gray")
            plt.title("no valid mouth crop")
        plt.axis("off")
        plot_idx += 1

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    cfg = Config()
    preview_all_train_videos(cfg)